## No langchain
## This has reranking, TSNE visualization and (query rewriting)

#### DO not run it with gemini because this will end your API limit

In [ ]:
from pathlib import Path
from openai import OpenAI
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from chromadb import PersistentClient
from tqdm import tqdm
import numpy as np
from sklearn.manifold import TSNE

load_dotenv(override=True)
MODEL = "gemini-3.5-flash-lite"
DB_NAME = 'preprocessed-db'
collection_name="docs"
embedding_model = "models/gemini-embedding-2-preview"
KNOWLEDGE_PATH = Path(r"C:\training projects\RAG\knowledge_base")
AVERAGE_CHUNK_SIZE = 500

In [ ]:
class Result(BaseModel):
    page_content:str
    metadata:dict

In [ ]:
class Chunk(BaseModel):
    headline:str=Field(description="A brief heading of the chunk a few words only")
    summary:str = Field(description="A few sentences summarizing content of this chunk")
    original_text: str=Field(description="Original text of this chunk exactly as is not changed at all")

    def as_result(self, document):
        metadata = {"source":document["source"], "type":document["type"]}
        return Result(page_content=self.headline+"\n\n"+self.summary+"\n\n"+self.original_text, metadata=metadata)

class Chunks(BaseModel):
    chunks:list[Chunk]


In [ ]:
def fetch_documents():
    """A homemade version of langchain directory loader"""
    documents=[]
    for folder in KNOWLEDGE_PATH.iterdir():
        doc_type = folder.name
        for file in folder.rglob("*.md"):
            with open(file, "r", encoding="utf-8") as f:
                documents.append({"type": doc_type, "source":file.as_posix(), "text": f.read()})
    print(f"Loaded {len(documents)} documents")
    return documents

In [ ]:
documents = fetch_documents()

# Make the chunks

In [ ]:
def make_prompt(document):
    how_many = (len(document["text"])//AVERAGE_CHUNK_SIZE)+1
    return f"""You take a document and split it into overlapping chunks for a Knowledgebase
    The document is from a a company 
    The document is of type:{document["type"]}
    The document has been retrieved from: {document["source"]}

    A chatbot will use chunks to answer question about company.
    You should divide the document as how you see fit and return in chunks do not leave anything
    The document should probably be split into {how_many} pieces but you can have more 
    or less as appropriate.
    There should be overlap between chunks about 25% overlap is expected.

    For each chunk you should provide headline, summary and original text of chunk.
    Together your chunk should represent entire document with overlap.
    Here is the document: {document["text"]}
    Responsd with chunks."""



In [ ]:
make_prompt(documents[0])

In [ ]:
def make_messages(document):
    return [
        {"role": "user", "content":make_prompt(document)}
    ]

In [ ]:
make_messages(documents[0])

In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
import os
llm = ChatGoogleGenerativeAI(
    temperature=0,  
    model=MODEL,
    seed=42,
    google_api_key=os.getenv("Gemini_API")
)


In [ ]:
def process_document(document):
    message = make_messages(document)
    from langchain_core.messages import SystemMessage, HumanMessage

    structured_llm = llm.with_structured_output(Chunks)

    response = structured_llm.invoke(message)
    reply = response[0]['text']
    doc_as_chunks = Chunks.model_validate_json(reply).chunks
    return [chunks.as_result(Document) for chunk in doc_as_chunks]

In [ ]:
process_document(documents[0])

In [ ]:
def create_chunks(documents):
    chunks=[]
    for doc in tqdm(documents):
        chunks.extend(process_document(doc))
    return chunks

# save the embeddings:

In [ ]:
def create_embeddings(chunks):
    chroma = PersistentClient(path=DB_NAME)
    if collection_name in [c.name for c in chroma.list_collections()]:
        chroma.delete_collections(collection_name)

    texts = [chunk.page_content for chunk in chunks]
    emb = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-2-preview",
    google_api_key=os.getenv("GEMINI_API_KEY")
    )

    vectors = [e.embeddings for e in emb]

    collection = chroma.get_or_create_collection(collection_name)
    ids = [str(i) for i in range(len(chunks))]
    metas = [chunk.metadata for chunk in chunks]

    collection.add(ids = ids, embeddings=vectors, documents=texts, metadata=metas)
    print(f"Vector store created with {collection.count()} documents")

## For visualization of RAG database if you ever need it--

In [ ]:
# chroma = PersistentClient(path=DB_NAME)
# collection = chroma.get_or_create_collection(collection_name)
# result = collection.get(include=['embeddings', 'documents', 'metadatas'])
# vectors = np.array(result['embeddings'])
# documents = result['documents']
# metadatas = result['metadatas']
# doc_types = [metadata['type'] for metadata in metadatas]
# colors = [['blue', 'green', 'red', 'orange'][['products', 'employees', 'contracts', 'company'].index(t)] for t in doc_types]
# tsne = TSNE(n_components=2, random_state=42)
# reduced_vectors = tsne.fit_transform(vectors)

# # Create the 2D scatter plot
# fig = go.Figure(data=[go.Scatter(
#     x=reduced_vectors[:, 0],
#     y=reduced_vectors[:, 1],
#     mode='markers',
#     marker=dict(size=5, color=colors, opacity=0.8),
#     text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
#     hoverinfo='text'
# )])

# fig.update_layout(title='2D Chroma Vector Store Visualization',
#     scene=dict(xaxis_title='x',yaxis_title='y'),
#     width=800,
#     height=600,
#     margin=dict(r=20, b=10, l=10, t=40)
# )

# fig.show()
# tsne = TSNE(n_components=3, random_state=42)
# reduced_vectors = tsne.fit_transform(vectors)

# # Create the 3D scatter plot
# fig = go.Figure(data=[go.Scatter3d(
#     x=reduced_vectors[:, 0],
#     y=reduced_vectors[:, 1],
#     z=reduced_vectors[:, 2],
#     mode='markers',
#     marker=dict(size=5, color=colors, opacity=0.8),
#     text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
#     hoverinfo='text'
# )])

# fig.update_layout(
#     title='3D Chroma Vector Store Visualization',
#     scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='z'),
#     width=900,
#     height=700,
#     margin=dict(r=10, b=10, l=10, t=40)
# )

# fig.show()

## Advanced RAG techniques :-
### Reranking
### and Query re-writing

In [ ]:
class RankOrder(BaseModel):
    order:list[int] = Field(description="""Order of relevance of chunks 
    from most relevant to least relevant""")

In [ ]:
def rerank(question, chunks):
    system_prompt="""You are a document re-ranker You have questions and list of relevant chunks of text
    from query of a knowledge base. The chunks are provided in the order they are retrieved this 
    should be apporximately ordered by relevance You must rank order the provided chunks by relevance 
    to the question with most relevant chunks first Reply only with list of ranked chunks ids nothing else
    Include all chunk ids you are provided with"""
    

In [ ]:
def rerank(question, chunks):
    system_prompt = """
You are a document re-ranker.
You are provided with a question and a list of relevant chunks of text from a query of a knowledge base.
The chunks are provided in the order they were retrieved; this should be approximately ordered by relevance, but you may be able to improve on that.
You must rank order the provided chunks by relevance to the question, with the most relevant chunk first.
Reply only with the list of ranked chunk ids, nothing else. Include all the chunk ids you are provided with, reranked.
"""
    user_prompt = f"The user has asked the following question:\n\n{question}\n\nOrder all the chunks of text by relevance to the question, from most relevant to least relevant. Include all the chunk ids you are provided with, reranked.\n\n"
    user_prompt += "Here are the chunks:\n\n"
    for index, chunk in enumerate(chunks):
        user_prompt += f"# CHUNK ID: {index + 1}:\n\n{chunk.page_content}\n\n"
    user_prompt += "Reply only with the list of ranked chunk ids, nothing else."
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]
    response = completion(model=MODEL, messages=messages, response_format=RankOrder)
    reply = response.choices[0].message.content
    order = RankOrder.model_validate_json(reply).order
    print(order)
    return [chunks[i - 1] for i in order]


RETRIEVAL_K = 10

def fetch_context_unranked(question):
    query = openai.embeddings.create(model=embedding_model, input=[question]).data[0].embedding
    results = collection.query(query_embeddings=[query], n_results=RETRIEVAL_K)
    chunks = []
    for result in zip(results["documents"][0], results["metadatas"][0]):
        chunks.append(Result(page_content=result[0], metadata=result[1]))
    return chunks

In [ ]:
question = "Who won the IIOTY award?"
chunks = fetch_context_unranked(question)
for chunk in chunks:
    print(chunk.page_content[:15]+"...")
reranked = rerank(question, chunks)

for chunk in reranked:
    print(chunk.page_content[:15]+"...")

In [ ]:
def fetch_context(question):
    chunks = fetch_context_unranked(question)
    return rerank(question, chunks)

In [ ]:
SYSTEM_PROMPT = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
Your answer will be evaluated for accuracy, relevance and completeness, so make sure it only answers the question and fully answers it.
If you don't know the answer, say so.
For context, here are specific extracts from the Knowledge Base that might be directly relevant to the user's question:
{context}

With this context, please answer the user's question. Be accurate, relevant and complete.
"""

def make_rag_messages(question, history, chunks):
    context = "\n\n".join(f"Extract from {chunk.metadata['source']}:\n{chunk.page_content}" for chunk in chunks)
    system_prompt = SYSTEM_PROMPT.format(context=context)
    return [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": question}]

def rewrite_query(question, history=[]):
    """Rewrite the user's question to be a more specific question that is more likely to surface relevant content in the Knowledge Base."""
    message = f"""
You are in a conversation with a user, answering questions about the company Insurellm.
You are about to look up information in a Knowledge Base to answer the user's question.

This is the history of your conversation so far with the user:
{history}

And this is the user's current question:
{question}

Respond only with a single, refined question that you will use to search the Knowledge Base.
It should be a VERY short specific question most likely to surface content. Focus on the question details.
Don't mention the company name unless it's a general question about the company.
IMPORTANT: Respond ONLY with the knowledgebase query, nothing else.
"""

    response = llm.invoke([{"role":"system", "content":message}])
    return response[0]['text']

In [ ]:
# rewrite_query("Who is the CTO of NovaMind Analytics?")

In [ ]:
def answer_question(question: str, history: list[dict] = []) -> tuple[str, list]:
    """
    Answer a question using RAG and return the answer and the retrieved context
    """
    query = rewrite_query(question, history)
    print(query)
    chunks = fetch_context(query)
    messages = make_rag_messages(question, history, chunks)
    response = completion(model=MODEL, messages=messages)
    return response.choices[0].message.content, chunks

answer_question("Who won the IIOTY award?", [])
answer_question("Who went to Manchester University?", [])